# 09 - Model Bias Monitoring (SageMaker Processing Job)

Runs a model bias monitor for the Yelp sentiment model as a **SageMaker
Processing Job**. The job compares a baseline window against a current
production window and checks whether model predictions differ materially
across a selected facet group, writing report artifacts back to S3.

This is the fairness/ethics evidence for the screencast and ties into the
monitoring reports built in `10_monitoring_reports.ipynb`, which reads this
job's `summary.json`.

**Prerequisites:** Run `04_split.ipynb` first. This notebook depends on:
- splits in `s3://<bucket>/splits/` (written by notebook 04)
- `project_config.json` (shared config used by notebooks 04-10)

**Design note:** The monitor scores both windows with a locally rebuilt copy of
the model (Random Forest, matching the best model from notebook 05), then runs
the bias computation inside a managed Processing Job. Low-level boto3 APIs are
used instead of the SageMaker Python SDK because some AWS Academy images ship an
incomplete `sagemaker` package.

## Monitor Design

- **Baseline window:** 2019 training split from `splits/train.parquet`
- **Current window:** 2020-2022 production split from `splits/production.parquet`
- **Model:** Random Forest sentiment classifier (matches notebook 05)
- **Target:** `sentiment` (1 = positive review sentiment)
- **Facet:** `review_length_group`, split at the baseline median review length
- **Protected/proxy group:** `short_review` · **Comparison group:** `long_review`

The dataset does not include demographic attributes such as gender, age, or
race. `review_length_group` is used as a **proxy facet** to demonstrate how the
bias monitor detects outcome differences between segments. In a production
system this would be replaced with a business-approved protected or
fairness-relevant facet.

## 0. Install Dependencies

In [3]:
import importlib
import subprocess
import sys


def install_if_missing(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        print(f"Installing {package}...")
        subprocess.run([sys.executable, "-m", "pip", "install", package, "--quiet"], check=True)
    else:
        print(f"{package} already installed")


# No SageMaker Python SDK dependency is required. This notebook uses low-level
# boto3 APIs because some AWS Academy images ship an incomplete sagemaker package.
for package, import_name in [
    ("boto3", "boto3"),
    ("pandas", "pandas"),
    ("pyarrow", "pyarrow"),
    ("scikit-learn", "sklearn"),
]:
    install_if_missing(package, import_name)

print("Dependencies ready")

boto3 already installed
pandas already installed
pyarrow already installed
scikit-learn already installed
Dependencies ready


## 1. Setup

Loads the shared config, resolves the region/bucket the project resources live
in, and creates the boto3 clients (S3, SageMaker, STS).

In [4]:
import json
import time
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse

import boto3
import numpy as np
import pandas as pd
from botocore.config import Config
from sklearn.ensemble import RandomForestClassifier


default_config = {
    "REGION": "us-east-1",
    "SOURCE_BUCKET": "aai540-group1-yelp-data",
    "FEATURE_COLS": ["review_length", "word_count", "useful", "funny", "cool", "vader_score"],
    "TARGET_COL": "sentiment",
    "RANDOM_STATE": 42,
}

config_path = Path("project_config.json")
if config_path.exists():
    with config_path.open() as f:
        cfg = {**default_config, **json.load(f)}
    print("Loaded project_config.json")
else:
    cfg = default_config.copy()
    print("project_config.json not found; using defaults")

REGION = cfg["REGION"]
SOURCE_BUCKET = cfg["SOURCE_BUCKET"]
FEATURE_COLS = cfg["FEATURE_COLS"]
TARGET_COL = cfg["TARGET_COL"]
RANDOM_STATE = cfg["RANDOM_STATE"]

aws_config = Config(connect_timeout=10, read_timeout=60, retries={"max_attempts": 2})
boto_session = boto3.Session(region_name=REGION)
s3 = boto_session.client("s3", config=aws_config)
sm = boto_session.client("sagemaker", config=aws_config)
sts = boto_session.client("sts", config=aws_config)

print("Region:", REGION)
print("Source bucket:", SOURCE_BUCKET)
print("Feature columns:", FEATURE_COLS)
print("Target column:", TARGET_COL)

Loaded project_config.json
Region: us-east-1
Source bucket: aai540-group1-yelp-data
Feature columns: ['review_length', 'word_count', 'useful', 'funny', 'cool', 'vader_score']
Target column: sentiment


In [5]:
def resolve_execution_role():
    identity = sts.get_caller_identity()
    arn = identity["Arn"]
    account = identity["Account"]
    if ":assumed-role/" in arn:
        role_name = arn.split(":assumed-role/")[1].split("/")[0]
        return f"arn:aws:iam::{account}:role/{role_name}"
    return arn


role = resolve_execution_role()
print("Execution role:", role)

Execution role: arn:aws:iam::476629097825:role/LabRole


## 2. Load the Baseline and Current Windows

Downloads the train (2019 baseline) and production (2020-2022 current) splits
from S3, using `SPLIT_PATHS` from the config when present.

In [6]:
def parse_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Expected s3:// URI, got {uri}")
    return parsed.netloc, parsed.path.lstrip("/")


def load_split(split_name):
    split_paths = cfg.get("SPLIT_PATHS", {})
    if split_name in split_paths:
        bucket, key = parse_s3_uri(split_paths[split_name])
    else:
        bucket, key = SOURCE_BUCKET, f"splits/{split_name}.parquet"

    local_path = Path("/tmp") / f"{split_name}.parquet"
    print(f"Downloading s3://{bucket}/{key} -> {local_path}")
    s3.download_file(bucket, key, str(local_path))
    return pd.read_parquet(local_path)


train_data = load_split("train")
prod_data = load_split("production")

print(f"Train rows: {len(train_data):,}")
print(f"Production rows: {len(prod_data):,}")

Train rows: 61,201
Production rows: 93,058


## 3. Train Model and Score Both Windows

Rebuilds the Random Forest, scores both windows, and assigns each row to a
facet group (`short_review` / `long_review`) split at the baseline median
review length. The scored frames are the inputs the Processing Job consumes.

In [7]:
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Training Random Forest model from the project train split...")
model.fit(train_data[FEATURE_COLS], train_data[TARGET_COL])
print("Model trained")

facet_column = "review_length"
facet_threshold = train_data[facet_column].median()


def score_for_monitor(df, window_name):
    scored = df.copy()
    probability = model.predict_proba(scored[FEATURE_COLS])[:, 1]
    scored["probability"] = probability
    scored["prediction"] = (probability >= 0.5).astype(int)
    scored["window"] = window_name
    scored["review_length_group"] = np.where(
        scored[facet_column] <= facet_threshold,
        "short_review",
        "long_review",
    )
    keep_cols = [
        "review_id",
        TARGET_COL,
        "prediction",
        "probability",
        "review_length_group",
        "window",
    ]
    return scored[[col for col in keep_cols if col in scored.columns]]


baseline_scored = score_for_monitor(train_data, "baseline_2019_train")
current_scored = score_for_monitor(prod_data, "current_2020_2022_production")

print("Baseline scored shape:", baseline_scored.shape)
print("Current scored shape:", current_scored.shape)
baseline_scored.head()

Training Random Forest model from the project train split...


Model trained


Baseline scored shape: (61201, 6)
Current scored shape: (93058, 6)


,review_id,sentiment,prediction,probability,review_length_group,window
0,YTZP0MR7U6KKS5xrjZOEdA,1,1,1.00,long_review,baseline_2019_train
1,rMNnC6Xu5qAiqRIAzwt4PA,1,1,0.98,long_review,baseline_2019_train
2,2ocF080JLEof22uZR2GUeA,1,1,1.00,short_review,baseline_2019_train
3,I_Kv6V-G0wmPq8fJDK--6g,0,0,0.04,short_review,baseline_2019_train
4,O4ZNLzFobs3xf8zA31DdUw,1,1,0.78,long_review,baseline_2019_train


## 4. Upload Processing Job Inputs to S3

Writes the scored baseline and current frames to a run-specific S3 prefix so
the Processing Job can read them.

In [8]:
run_id = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%S")
processing_prefix = f"bias-monitor/processing-runs/{run_id}"
input_prefix = f"{processing_prefix}/input"
output_prefix = f"{processing_prefix}/output"

local_input_dir = Path("/tmp/bias-monitor-input")
local_input_dir.mkdir(parents=True, exist_ok=True)

baseline_local = local_input_dir / "baseline.csv"
current_local = local_input_dir / "current.csv"
baseline_scored.to_csv(baseline_local, index=False)
current_scored.to_csv(current_local, index=False)

baseline_s3_uri = f"s3://{SOURCE_BUCKET}/{input_prefix}/baseline.csv"
current_s3_uri = f"s3://{SOURCE_BUCKET}/{input_prefix}/current.csv"
output_s3_uri = f"s3://{SOURCE_BUCKET}/{output_prefix}"

s3.upload_file(str(baseline_local), SOURCE_BUCKET, f"{input_prefix}/baseline.csv")
s3.upload_file(str(current_local), SOURCE_BUCKET, f"{input_prefix}/current.csv")

print("Uploaded Processing Job inputs:")
print("Baseline:", baseline_s3_uri)
print("Current :", current_s3_uri)
print("Output  :", output_s3_uri)

Uploaded Processing Job inputs:
Baseline: s3://aai540-group1-yelp-data/bias-monitor/processing-runs/2026-06-06-00-07-04/input/baseline.csv
Current : s3://aai540-group1-yelp-data/bias-monitor/processing-runs/2026-06-06-00-07-04/input/current.csv
Output  : s3://aai540-group1-yelp-data/bias-monitor/processing-runs/2026-06-06-00-07-04/output


## 5. Write the Bias Computation Script

Generates the script the Processing Job runs in-container. It computes
per-group classification metrics, derives the bias metrics (disparate impact,
demographic parity, false-negative-rate difference), applies the thresholds,
and writes the report artifacts (including `summary.json`).

In [9]:
scripts_dir = Path("scripts")
scripts_dir.mkdir(exist_ok=True)
processor_script = scripts_dir / "bias_monitor_processor.py"

processor_script.write_text(
    r'''
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd


def safe_divide(numerator, denominator):
    return np.nan if denominator == 0 else numerator / denominator


def group_classification_metrics(group_df, target_col):
    y_true = group_df[target_col].astype(int)
    y_pred = group_df["prediction"].astype(int)

    positives = y_true == 1
    negatives = y_true == 0

    return pd.Series(
        {
            "rows": len(group_df),
            "positive_prediction_rate": y_pred.mean(),
            "actual_positive_rate": y_true.mean(),
            "accuracy": (y_true == y_pred).mean(),
            "false_positive_rate": safe_divide(((y_pred == 1) & negatives).sum(), negatives.sum()),
            "false_negative_rate": safe_divide(((y_pred == 0) & positives).sum(), positives.sum()),
            "average_probability": group_df["probability"].mean(),
        }
    )


def metrics_by_group(df, facet_col, target_col):
    rows = []
    for group_name, group_df in df.groupby(facet_col, observed=True):
        metrics = group_classification_metrics(group_df, target_col)
        metrics.name = group_name
        rows.append(metrics)
    return pd.DataFrame(rows)


def bias_metrics(group_metrics, protected_group, comparison_group):
    protected_rate = group_metrics.loc[protected_group, "positive_prediction_rate"]
    comparison_rate = group_metrics.loc[comparison_group, "positive_prediction_rate"]
    protected_fnr = group_metrics.loc[protected_group, "false_negative_rate"]
    comparison_fnr = group_metrics.loc[comparison_group, "false_negative_rate"]

    return {
        "protected_positive_prediction_rate": protected_rate,
        "comparison_positive_prediction_rate": comparison_rate,
        "demographic_parity_difference": protected_rate - comparison_rate,
        "disparate_impact_ratio": safe_divide(protected_rate, comparison_rate),
        "false_negative_rate_difference": protected_fnr - comparison_fnr,
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--baseline", default="/opt/ml/processing/input/baseline/baseline.csv")
    parser.add_argument("--current", default="/opt/ml/processing/input/current/current.csv")
    parser.add_argument("--output", default="/opt/ml/processing/output")
    parser.add_argument("--target-col", default="sentiment")
    parser.add_argument("--facet-col", default="review_length_group")
    parser.add_argument("--protected-group", default="short_review")
    parser.add_argument("--comparison-group", default="long_review")
    parser.add_argument("--min-disparate-impact-ratio", type=float, default=0.80)
    parser.add_argument("--max-abs-demographic-parity-difference", type=float, default=0.10)
    parser.add_argument("--max-false-negative-rate-difference", type=float, default=0.10)
    parser.add_argument("--max-disparate-impact-shift", type=float, default=0.15)
    args = parser.parse_args()

    output_dir = Path(args.output)
    output_dir.mkdir(parents=True, exist_ok=True)

    baseline = pd.read_csv(args.baseline)
    current = pd.read_csv(args.current)

    baseline_group_metrics = metrics_by_group(baseline, args.facet_col, args.target_col)
    current_group_metrics = metrics_by_group(current, args.facet_col, args.target_col)

    baseline_bias = bias_metrics(baseline_group_metrics, args.protected_group, args.comparison_group)
    current_bias = bias_metrics(current_group_metrics, args.protected_group, args.comparison_group)

    bias_comparison = pd.DataFrame([baseline_bias, current_bias], index=["baseline", "current"])
    bias_comparison.loc["change"] = bias_comparison.loc["current"] - bias_comparison.loc["baseline"]

    violations = []
    if current_bias["disparate_impact_ratio"] < args.min_disparate_impact_ratio:
        violations.append(
            {
                "metric": "disparate_impact_ratio",
                "value": current_bias["disparate_impact_ratio"],
                "threshold": args.min_disparate_impact_ratio,
                "message": "Protected group receives positive predictions at less than 80% of comparison group rate.",
            }
        )

    if abs(current_bias["demographic_parity_difference"]) > args.max_abs_demographic_parity_difference:
        violations.append(
            {
                "metric": "demographic_parity_difference",
                "value": current_bias["demographic_parity_difference"],
                "threshold": args.max_abs_demographic_parity_difference,
                "message": "Positive prediction rates differ by more than 10 percentage points across groups.",
            }
        )

    if current_bias["false_negative_rate_difference"] > args.max_false_negative_rate_difference:
        violations.append(
            {
                "metric": "false_negative_rate_difference",
                "value": current_bias["false_negative_rate_difference"],
                "threshold": args.max_false_negative_rate_difference,
                "message": "Protected group has materially higher false negative rate.",
            }
        )

    impact_shift = current_bias["disparate_impact_ratio"] - baseline_bias["disparate_impact_ratio"]
    if abs(impact_shift) > args.max_disparate_impact_shift:
        violations.append(
            {
                "metric": "disparate_impact_ratio_shift",
                "value": impact_shift,
                "threshold": args.max_disparate_impact_shift,
                "message": "Disparate impact ratio shifted materially from baseline to current window.",
            }
        )

    violations_df = pd.DataFrame(violations)

    baseline_group_metrics.to_csv(output_dir / "baseline_group_metrics.csv")
    current_group_metrics.to_csv(output_dir / "current_group_metrics.csv")
    bias_comparison.to_csv(output_dir / "bias_metric_comparison.csv")
    violations_df.to_csv(output_dir / "bias_violations.csv", index=False)

    summary = {
        "status": "violations_detected" if violations else "pass",
        "violation_count": len(violations),
        "facet_column": args.facet_col,
        "protected_group": args.protected_group,
        "comparison_group": args.comparison_group,
        "baseline_rows": int(len(baseline)),
        "current_rows": int(len(current)),
        "current_bias_metrics": current_bias,
    }
    with (output_dir / "summary.json").open("w") as f:
        json.dump(summary, f, indent=2)

    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()
'''.strip()
)

print(f"Wrote processing script: {processor_script}")

Wrote processing script: scripts/bias_monitor_processor.py


## 6. Run the SageMaker Processing Job

Launches a managed Processing Job with the scikit-learn image, waits for it to
complete, and fails loudly if the job does not reach `Completed`.

In [10]:
processing_job_name = f"yelp-bias-monitor-{run_id}".replace("_", "-")
script_s3_key = f"{processing_prefix}/code/{processor_script.name}"
s3.upload_file(str(processor_script), SOURCE_BUCKET, script_s3_key)
script_s3_uri = f"s3://{SOURCE_BUCKET}/{script_s3_key}"

# Scikit-learn Processing image for us-east-1. This is the same image family
# used by the project training notebooks, but launched through boto3 directly.
if REGION != "us-east-1":
    raise ValueError("This notebook currently hardcodes the sklearn image for us-east-1.")
processing_image_uri = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

print("Creating Processing Job:", processing_job_name)
print("Script:", script_s3_uri)
print("Image:", processing_image_uri)

sm.create_processing_job(
    ProcessingJobName=processing_job_name,
    RoleArn=role,
    AppSpecification={
        "ImageUri": processing_image_uri,
        "ContainerEntrypoint": ["python", "/opt/ml/processing/code/bias_monitor_processor.py"],
        "ContainerArguments": [
            "--target-col", TARGET_COL,
            "--facet-col", "review_length_group",
            "--protected-group", "short_review",
            "--comparison-group", "long_review",
        ],
    },
    ProcessingInputs=[
        {
            "InputName": "baseline",
            "S3Input": {
                "S3Uri": baseline_s3_uri,
                "LocalPath": "/opt/ml/processing/input/baseline",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "S3DataDistributionType": "FullyReplicated",
                "S3CompressionType": "None",
            },
        },
        {
            "InputName": "current",
            "S3Input": {
                "S3Uri": current_s3_uri,
                "LocalPath": "/opt/ml/processing/input/current",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "S3DataDistributionType": "FullyReplicated",
                "S3CompressionType": "None",
            },
        },
        {
            "InputName": "code",
            "S3Input": {
                "S3Uri": script_s3_uri,
                "LocalPath": "/opt/ml/processing/code",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "S3DataDistributionType": "FullyReplicated",
                "S3CompressionType": "None",
            },
        },
    ],
    ProcessingOutputConfig={
        "Outputs": [
            {
                "OutputName": "bias-monitor-report",
                "S3Output": {
                    "S3Uri": output_s3_uri,
                    "LocalPath": "/opt/ml/processing/output",
                    "S3UploadMode": "EndOfJob",
                },
            }
        ]
    },
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.m5.large",
            "VolumeSizeInGB": 20,
        }
    },
    StoppingCondition={"MaxRuntimeInSeconds": 1800},
)

while True:
    desc = sm.describe_processing_job(ProcessingJobName=processing_job_name)
    status = desc["ProcessingJobStatus"]
    print("Processing status:", status)
    if status in ["Completed", "Failed", "Stopped"]:
        break
    time.sleep(30)

if status != "Completed":
    raise RuntimeError(desc.get("FailureReason", f"Processing job ended with status {status}"))

print("Processing job completed:", processing_job_name)
print("Report output:", output_s3_uri)

Creating Processing Job: yelp-bias-monitor-2026-06-06-00-07-04
Script: s3://aai540-group1-yelp-data/bias-monitor/processing-runs/2026-06-06-00-07-04/code/bias_monitor_processor.py
Image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


Processing status: InProgress


Processing status: InProgress


Processing status: InProgress


Processing status: InProgress


...

Processing status: InProgress


Processing status: InProgress


Processing status: InProgress


Processing status: InProgress


Processing status: Completed
Processing job completed: yelp-bias-monitor-2026-06-06-00-07-04
Report output: s3://aai540-group1-yelp-data/bias-monitor/processing-runs/2026-06-06-00-07-04/output


## 7. Download Report Artifacts

Pulls the report artifacts the job wrote to S3 back to the notebook for
inspection.

In [11]:
local_report_dir = Path("bias_monitor_processing_outputs") / run_id
local_report_dir.mkdir(parents=True, exist_ok=True)

report_files = [
    "summary.json",
    "baseline_group_metrics.csv",
    "current_group_metrics.csv",
    "bias_metric_comparison.csv",
    "bias_violations.csv",
]

for filename in report_files:
    s3.download_file(SOURCE_BUCKET, f"{output_prefix}/{filename}", str(local_report_dir / filename))

print("Downloaded report artifacts:")
for path in sorted(local_report_dir.iterdir()):
    print("-", path)

Downloaded report artifacts:
- bias_monitor_processing_outputs/2026-06-06-00-07-04/baseline_group_metrics.csv
- bias_monitor_processing_outputs/2026-06-06-00-07-04/bias_metric_comparison.csv
- bias_monitor_processing_outputs/2026-06-06-00-07-04/bias_violations.csv
- bias_monitor_processing_outputs/2026-06-06-00-07-04/current_group_metrics.csv
- bias_monitor_processing_outputs/2026-06-06-00-07-04/summary.json


## 8. Results

Summarizes the monitor verdict and shows the baseline-vs-current bias metric
comparison.

In [12]:
with (local_report_dir / "summary.json").open() as f:
    summary = json.load(f)

print("SAGEMAKER PROCESSING BIAS MONITOR SUMMARY")
print("=" * 48)
print("Status:", summary["status"])
print("Violation count:", summary["violation_count"])
print("Facet column:", summary["facet_column"])
print("Protected group:", summary["protected_group"])
print("Comparison group:", summary["comparison_group"])
print("Baseline rows:", f"{summary['baseline_rows']:,}")
print("Current rows:", f"{summary['current_rows']:,}")
print("\nCurrent bias metrics:")
for metric, value in summary["current_bias_metrics"].items():
    print(f"- {metric}: {value:.4f}")

pd.read_csv(local_report_dir / "bias_violations.csv")

SAGEMAKER PROCESSING BIAS MONITOR SUMMARY
Status: violations_detected
Violation count: 1
Facet column: review_length_group
Protected group: short_review
Comparison group: long_review
Baseline rows: 61,201
Current rows: 93,058

Current bias metrics:
- protected_positive_prediction_rate: 0.8140
- comparison_positive_prediction_rate: 0.6524
- demographic_parity_difference: 0.1616
- disparate_impact_ratio: 1.2478
- false_negative_rate_difference: -0.0294


,metric,value,threshold,message
0,demographic_parity_difference,0.161647,0.1,Positive prediction rates differ by more than ...


In [13]:
pd.read_csv(local_report_dir / "bias_metric_comparison.csv", index_col=0)

,protected_positive_prediction_rate,comparison_positive_prediction_rate,demographic_parity_difference,disparate_impact_ratio,false_negative_rate_difference
baseline,0.809774,0.659613,0.150161,1.227650,0.000684
current,0.814041,0.652394,0.161647,1.247776,-0.029382
change,0.004267,-0.007219,0.011486,0.020126,-0.030065


## 9. Interpretation

The SageMaker Processing Job reads scored prediction data from S3, computes bias metrics, and writes report artifacts back to S3.

The main metrics are:

- **Disparate impact ratio** — ratio of favorable-outcome rates between groups (four-fifths rule: flag below 0.80)
- **Demographic parity difference** — gap in positive-prediction rates across groups (flag above 0.10)
- **False negative rate difference** — gap in missed-positive rates across groups (flag above 0.10)

If any metric crosses the configured threshold, the monitor reports a bias violation. The latest `summary.json` from this job is read back and embedded in the monitoring reports built by `10_monitoring_reports.ipynb`.